# GridapTensorProduct.jl — Library Walk-Through

This notebook traces the **five-stage pipeline** of GridapTensorProduct.jl from raw 1D
meshes all the way to a solved linear system, explaining how each structure is built and
how the stages connect.

**Domain:** 2D Poisson on Ω = [0,1]² with homogeneous Dirichlet BCs and unit forcing:
$$-\Delta u = 1 \quad \text{in } \Omega, \qquad u = 0 \quad \text{on } \partial\Omega$$

---
## Architecture overview

```
Stage 1  TensorProductMeasure        — bundle N subdomain quadrature measures
Stage 2  TensorProductFESpace        — bundle N subdomain FE spaces + DOF mapping
Stage 3  TensorProductGeometry       — triangulation and cell-field wrappers
Stage 4  TensorProductFEMOperators   — extract 1D matrices; form Kronecker global ops
Stage 5  TensorProductAssembly       — high-level API; assemble and return (A, b)
```

Key idea: **work with 1D subdomains independently in Stages 1–4; couple via Kronecker products only at assembly (Stage 5).**

In [ ]:
using Gridap
using GridapTensorProduct
import GridapTensorProduct: ⊗
using LinearAlgebra

---
## Stage 1 — 1D meshes and subdomain measures

Everything starts with ordinary Gridap objects on **1D subdomains**. No tensor-product types yet.

In [ ]:
Nx, Ny = 4, 3
model_x = CartesianDiscreteModel((0.0, 1.0), Nx)
model_y = CartesianDiscreteModel((0.0, 1.0), Ny)
Ωx = Interior(model_x)
Ωy = Interior(model_y)

quad_order = 2
dΩx = Measure(Ωx, quad_order)
dΩy = Measure(Ωy, quad_order)

println("Subdomain x: ", num_cells(Ωx), " cells")
println("Subdomain y: ", num_cells(Ωy), " cells")

### TensorProductMeasure — the ⊗ operator

Writing `dΩx ⊗ dΩy` creates a `TensorProductMeasure` that bundles the two measures.
When a user writes `∫(expr) * (dΩx ⊗ dΩy)`, the library intercepts it and creates
a `TensorProductIntegrand` marker — no actual integration happens at this point.
Assembly is deferred to Stage 5.

In [ ]:
dΩ_tp = dΩx ⊗ dΩy

println(typeof(dΩ_tp))           # TensorProductMeasure
println(num_subdomains(dΩ_tp))   # 2

# Intercepted multiplication creates a deferred marker
marker = ∫(x -> 1.0) * dΩ_tp
println(typeof(marker))           # TensorProductIntegrand

---
## Stage 2 — Subdomain FE spaces and the tensor DOF map

Each subdomain gets its own standard Gridap `TestFESpace` / `TrialFESpace`.
Dirichlet BCs are applied **per subdomain** before any tensor coupling.

**DOF ordering convention** (subdomain 1 = fastest-varying):
$$\text{global\_dof}(i_1, i_2) = i_1 + (i_2 - 1) \cdot n_{f,1}$$

This matches the Kronecker convention `kron(M_2, M_1)` where M_1 is innermost.

In [ ]:
reffex = ReferenceFE(lagrangian, Float64, 1)
reffey = ReferenceFE(lagrangian, Float64, 2)

Vx = TestFESpace(Ωx, reffex; conformity=:H1, dirichlet_tags="boundary")
Vy = TestFESpace(Ωy, reffey; conformity=:H1, dirichlet_tags="boundary")
Ux = TrialFESpace(Vx, 0.0)
Uy = TrialFESpace(Vy, 0.0)

println("Free DOFs on Ωx: ", num_free_dofs(Vx))   
println("Free DOFs on Ωy: ", num_free_dofs(Vy))   

In [ ]:
V_tp = TensorProductFESpace(Vx, Vy)
U_tp = TensorProductFESpace(Ux, Uy)

n_tp = num_free_dofs(V_tp)   # = num_free_dofs(Vx) * num_free_dofs(Vy)
println("Free DOFs: ", num_free_dofs(Vx), " × ", num_free_dofs(Vy), " = ", n_tp)

# Satisfies Gridap's FESpace interface
println(get_free_dof_ids(V_tp))   # Base.OneTo(49)

In [ ]:
get_cell_ids(V_tp)   # Vector of cell IDs build as tuples of cell ids from each subdomain

In [ ]:
get_cell_dof_ids(V_tp)   # Vector of vectors of DOF IDs for each cell, with DOF IDs as tuples of DOF ids from each subdomain

---
## Stage 3 — Tensor-product geometry

`TensorProductTriangulation` wraps N component triangulations.
Its `num_cells` equals the product of component cell counts.
Access individual components with integer indexing.

In [ ]:
trian_tp = get_triangulation(V_tp)

println(typeof(trian_tp))                             # TensorProductTriangulation
println("Total tensor cells: ", num_cells(trian_tp))  # 8 * 8 = 64
println("Cells on Ωx: ",        num_cells(trian_tp[1]))
println("Cells on Ωy: ",        num_cells(trian_tp[2]))

# Plural accessor for direct tuple access
trians = get_triangulations(V_tp)
println("Number of components: ", length(trians))

---
## Stage 4 — Subdomain operator extraction and Kronecker assembly

### 4a: Assemble fundamental 1D matrices

For each subdomain $k$, `assemble_subdomain_operators` calls Gridap's
`AffineFEOperator` to extract five matrices:

| Matrix | Weak form |
|--------|----------|
| $M^{(k)}$ | $\int_{\Omega_k} \varphi_i \varphi_j \, dx_k$ |
| $K^{(k)}$ | $\int_{\Omega_k} \nabla\varphi_i \cdot \nabla\varphi_j \, dx_k$ |
| $G^{(k)}$ | $\int_{\Omega_k} (\partial_x \varphi_i) \, \varphi_j \, dx_k$ |
| $D^{(k)}$ | $\int_{\Omega_k} \varphi_i \, (\partial_x \varphi_j) \, dx_k$ |
| $A^{(k)}$ | $\int_{\Omega_k} \varphi_i \, (b_k \partial_x \varphi_j) \, dx_k$ |

In [ ]:
ops = assemble_subdomain_operators((Vx, Vy), (dΩx, dΩy))

println(typeof(ops))   # TensorProductSubdomainOperators{2}
println("M^(x) size: ", size(ops.M_ops[1]))  
println("K^(x) size: ", size(ops.K_ops[1]))  
println("M^(y) size: ", size(ops.M_ops[2]))  
println("K^(y) size: ", size(ops.K_ops[2])) 

### 4b: Kronecker assembly

Global operators follow the **subdomain 1 = innermost** convention:

| Operator | N=2 formula |
|----------|-------------|
| Mass | `kron(My, Mx)` |
| Stiffness | `kron(My, Kx) + kron(Ky, Mx)` |
| Gradient | `[kron(My, Gx); kron(Gy, Mx)]` (block column) |

In [ ]:
M_global = assemble_mass_tensor(ops)
A_global = assemble_stiffness_tensor(ops)
G_global = assemble_gradient_tensor(ops)

println("Global mass size:      ", size(M_global))  # (49, 49)
println("Global stiffness size: ", size(A_global))  # (49, 49)
println("Global gradient size:  ", size(G_global))  # (98, 49) — 2 blocks stacked
println("Stiffness symmetric:   ", maximum(abs, A_global - A_global') < 1e-14)

# Verify against direct kron
Mx, Kx = ops.M_ops[1], ops.K_ops[1]
My, Ky = ops.M_ops[2], ops.K_ops[2]
A_ref = kron(My, Kx) + kron(Ky, Mx)
println("Max error vs kron reference: ", maximum(abs, A_global - A_ref))

### Low-level API: direct subdomain matrix access

The six Kronecker assemblers can be called directly from `TensorProductSubdomainOperators`.
Each matrix type (`M`, `K`, `G`, `D`, `A`) is assembled lazily and cached on first access.


In [ ]:
# Direct Kronecker assembler calls (all return Matrix{Float64})
M_global = assemble_mass_tensor(ops)
A_stiff  = assemble_stiffness_tensor(ops)

# Verify stiffness against manual kron
Mx, Kx = ops.M_ops[1], ops.K_ops[1]
My, Ky = ops.M_ops[2], ops.K_ops[2]
A_ref = kron(My, Kx) + kron(Ky, Mx)
println("Max error vs manual kron: ", maximum(abs, A_stiff - A_ref))  # 0.0

---
## Stage 5 — High-level assembly: `TensorProductAffineOperator`

The recommended entry point for most users. Provide **labeled contributions** for each
bilinear and linear form term. The assembler dispatches to the correct Kronecker formula
based on the label — no heuristic classification.

### Type hierarchy (top = containing)

```
TensorProductAffineOperator
  └── TensorProductOperator  (lazy, caches A and b)
        └── TensorProductWeakForm
              ├── lhs_terms :: Vector{TensorProductDomainContribution}
              └── rhs_terms :: Vector{TensorProductLinearContribution}
```

### Valid labels

| Label | Subdomain ops assembled | Kronecker formula |
|:------|:------------------------|:------------------|
| `:mass` | M_k | `M₁ ⊗ M₂ ⊗ … ⊗ Mₙ` |
| `:stiffness` | M_k, K_k | `Σₖ M⊗…⊗Kₖ⊗…⊗M` |
| `:gradient` | M_k, G_k | block-column |
| `:divergence` | M_k, G_k | block-row (Gᵀ) |
| `:curl_curl` | M_k, D_k | double sum |
| `:advection` | M_k, A_k | `Σₖ bₖ·M⊗…⊗Aₖ⊗…⊗M` |


### Poisson equation

Single stiffness term: `a(u,v) = ∫(∇u·∇v) dΩ`, unit forcing `f=1`.

In [ ]:
lhs = [TensorProductDomainContribution(:stiffness, dΩ_tp)]
rhs = [TensorProductLinearContribution(dΩ_tp)]
wf  = TensorProductWeakForm(lhs, rhs)

op  = TensorProductAffineOperator(wf, V_tp, U_tp)
A   = get_matrix(op)   # triggers assembly on first call
b   = get_vector(op)
u   = A \ b

println("A size:       ", size(A))
println("b norm:       ", round(norm(b), digits=6))
println("Max solution: ", round(maximum(u), digits=6))

### Helmholtz equation: K + k²M

Multiple LHS contributions are **accumulated** automatically.
Each term carries its own scalar `coefficient`.

In [ ]:
κ² = 4.0

lhs_helm = [
    TensorProductDomainContribution(:stiffness, dΩ_tp),
    TensorProductDomainContribution(:mass,      dΩ_tp; coefficient = κ²)
]
op_helm = TensorProductAffineOperator(TensorProductWeakForm(lhs_helm, rhs), V_tp, U_tp)
A_helm  = get_matrix(op_helm)

# Verify against manual Kronecker
A_ref_helm = assemble_stiffness_tensor(ops) .+ κ² .* assemble_mass_tensor(ops)
println("Helmholtz max error: ", maximum(abs, A_helm - A_ref_helm))  # 0.0
println("Symmetric: ", maximum(abs, A_helm - A_helm') < 1e-12)

### Advection–diffusion

Separable advection requires the per-direction velocity components `b_coeffs = (b₁, b₂, …)`.
The assembler builds `A⁽ᵏ⁾ = ∫ φᵢ (bₖ ∂φⱼ/∂xₖ) dxₖ` per subdomain.


In [ ]:
bx, by = 2.0, 1.0

lhs_adv = [
    TensorProductDomainContribution(:stiffness, dΩ_tp),
    TensorProductDomainContribution(:advection, dΩ_tp; b_coeffs = (bx, by))
]
op_adv  = TensorProductAffineOperator(TensorProductWeakForm(lhs_adv, rhs), V_tp, U_tp)
A_adv   = get_matrix(op_adv)

println("Advection-diffusion A size: ", size(A_adv))
println("Asymmetric (expected):      ", maximum(abs, A_adv - A_adv') > 1e-10)

---
## Custom separable RHS with `rhs_forms`

For separable non-unit forcing `f(x,y) = f₁(x)·f₂(y)`, supply per-subdomain
1D linear forms via `rhs_forms`. The global vector is assembled as `kron(b₂, b₁)`.

In [ ]:
f_x(x) = x[1]
f_y(y) = y[1]

rhs_custom = [TensorProductLinearContribution(dΩ_tp;
    rhs_forms = (v -> ∫(f_x * v) * dΩx,
                 v -> ∫(f_y * v) * dΩy))]

op_custom = TensorProductAffineOperator(TensorProductWeakForm(lhs, rhs_custom), V_tp, U_tp)
b_custom  = get_vector(op_custom)

println("Custom RHS norm: ", round(norm(b_custom), digits=6))
println("Unit RHS norm:   ", round(norm(b), digits=6))  # should differ

---
## Functional shortcut API

`assemble_tensor_affine_operator` is a convenience wrapper for the single-term case.
It builds the `TensorProductWeakForm` internally.

In [ ]:
# Equivalent to the Poisson example above
A2, b2 = assemble_tensor_affine_operator(:stiffness, dΩ_tp, U_tp, V_tp)

println("A2 == A: ", maximum(abs, A2 - A) == 0.0)
println("b2 == b: ", maximum(abs, b2 - b) == 0.0)

# Single-term mass
A_m, _ = assemble_tensor_affine_operator(:mass, dΩ_tp, U_tp, V_tp)
println("Mass max error: ", maximum(abs, A_m - assemble_mass_tensor(ops)))  # 0.0

---
## Extension to 3D: add a third subdomain

The API is **dimension-agnostic**. N=3 requires no code changes beyond
adding a third subdomain space and building a 3-component `TensorProductMeasure`.

In [ ]:
reffez = ReferenceFE(lagrangian, Float64, 1)
model_z = CartesianDiscreteModel((0.0, 1.0), 5)
Ωz  = Interior(model_z)
dΩz = Measure(Ωz, quad_order)
Vz  = TestFESpace(Ωz, reffez; conformity=:H1, dirichlet_tags="boundary")
Uz  = TrialFESpace(Vz, 0.0)

V3 = TensorProductFESpace(Vx, Vy, Vz)
U3 = TensorProductFESpace(Ux, Uy, Uz)

dΩ_tp3 = dΩx ⊗ dΩy ⊗ dΩz   # TensorProductMeasure{3}

lhs3 = [TensorProductDomainContribution(:stiffness, dΩ_tp3)]
rhs3 = [TensorProductLinearContribution(dΩ_tp3)]
op3  = TensorProductAffineOperator(TensorProductWeakForm(lhs3, rhs3), V3, U3)

A3 = get_matrix(op3)
b3 = get_vector(op3)

println("3D system size: ", size(A3))     # (nf_x * nf_y * nf_z)²
println("3D symmetric:   ", maximum(abs, A3 - A3') < 1e-13)

---
## Connection map

```
CartesianDiscreteModel (×N 1D meshes)
       │
       ▼
 Measure(Ωk, q)  ──⊗──►  TensorProductMeasure            [Stage 1]
       │
       ▼
 TestFESpace / TrialFESpace  ──►  TensorProductFESpace     [Stage 2]
       │                          num_free_dofs = ∏ nf_k
       │
       ▼
 get_triangulation  ──►  TensorProductTriangulation        [Stage 3]
       │                   num_cells = ∏ nc_k
       │
       ▼
 assemble_subdomain_operators  ──►  SubdomainOperators{N}  [Stage 4a]
       │                             M, K, G, D, A per subdomain (lazy)
       ▼
 assemble_stiffness_tensor (etc.)                          [Stage 4b]
       │   A = ∑_k kron(M_N,…,K_k,…,M_1)
       │
       ▼
 TensorProductDomainContribution(:stiffness, dΩ_tp)        [Stage 5 input]
 TensorProductLinearContribution(dΩ_tp)
 TensorProductWeakForm(lhs, rhs)
 TensorProductAffineOperator(wf, V_tp, U_tp)
       │
       ▼  get_matrix(op) / get_vector(op)
       │
       ▼
       A \ b  →  u_dofs
```
